# DiploMind: 3×3 dataset generation (language × depth)

Each article (Urdu or English) becomes **9 training rows**: 3 output languages (`en`, `ur`, `urdish`) × 3 depths (`easy`, `medium`, `hard`).

**Pipeline** (one model for every step: DeepSeek V4.1-Flash)
1. **Input:** cleaned article paragraphs, 150–900 words, about 30% starting mid-article. This is the training input.
2. **Facts:** input → a list of short Urdu facts + 0–3 terms a beginner can't guess from the words (internal step only).
3. **Urdu depths:** facts → Urdu easy / medium / hard.
4. **English:** Urdu → English, told which reader it writes for. For English articles, the original is given as a reference for names and terms.
5. **Urdish:** Urdu + English → Urdish (Urdu sentences in Roman letters, with common English words mixed in).

**Depth = IR knowledge assumed, never language difficulty.** Easy explains key terms, actors and one background sentence; medium explains the 1–2 most important concepts briefly; hard explains nothing. All three stay easy to read.

**3 machines:** set a different `MACHINE_ID` on each (section 3). Re-running section 11 resumes where it stopped.

## 1. Install

In [2]:
!pip install -q openai ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 525.5 kB/s eta 0:00:000:00:01


## 2. API key
Paste your W&B key below, or on Kaggle add a secret named `WANDB_API_KEY` (Add-ons → Secrets). Clear the key before sharing the notebook.

In [3]:
WANDB_API_KEY = ""   # paste your key here, or use a Kaggle secret
WANDB_PROJECT = "m-ibrahim094131-fast-nuces/test"

import os
if not WANDB_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
print("API key set" if WANDB_API_KEY else "No API key yet: paste it above")

API key set


## 3. Settings

In [4]:
import os, re, csv, json, time, glob, random, hashlib, threading, collections
from concurrent.futures import ThreadPoolExecutor

MACHINE_ID    = "qt_test"   # m1 / m2 / m3: different on every machine
ARTICLES_DIRS = [
    "/kaggle/input/datasets/mibrahim094131/urdu123",   # Urdu articles
    "/kaggle/input/datasets/mibrahim094131/english123",                     # English articles (folder with bbc_*.jsonl)
]
OUT_ROOT = "dataset_out"

MODEL    = "deepseek-ai/DeepSeek-V4.1-Flash"   # used for every step
BASE_URL = "https://api.inference.wandb.ai/v1"

LANGS  = ["en", "urdish", "ur"]
DEPTHS = ["easy", "medium", "hard"]

MAX_ARTICLES = 40     # per folder; None = every article
WORKERS      = 4      # articles in parallel; lower to 2 if you see [llm retry] lines
EXAMPLE_RATE = 0.3    # share of articles whose EASY cells get one everyday example
TEMPERATURE  = 0.3
INPUT_MIN, INPUT_MAX = 150, 900   # training input length in words; match what retrieval returns

# Optional automatic split across machines. Leave NUM_SHARDS = 1 if each machine has its own folders.
NUM_SHARDS  = 1
SHARD_INDEX = 0

OUT_DIR = os.path.join(OUT_ROOT, MACHINE_ID)
os.makedirs(OUT_DIR, exist_ok=True)
RAW_PATH  = os.path.join(OUT_DIR, f"grid_raw_v3_{MACHINE_ID}.jsonl")       # every processed article (resume file)
PREP_PATH = os.path.join(OUT_DIR, f"prep_cache_mixed_{MACHINE_ID}.jsonl")  # cached fact lists
print("Machine:", MACHINE_ID, "| model:", MODEL, "| writing to", OUT_DIR)

Machine: qt_test | model: deepseek-ai/DeepSeek-V4.1-Flash | writing to dataset_out/qt_test


## 4. Model client

In [5]:
from openai import OpenAI

assert WANDB_API_KEY, "Paste the API key in section 2 first."
client = OpenAI(base_url=BASE_URL, api_key=WANDB_API_KEY, project=WANDB_PROJECT)

def is_reasoning(m):   # thinking models need a bigger token budget
    return any(k in m.lower() for k in ("gpt-oss", "kimi", "glm", "minimax", "qwen3", "deepseek-v4"))

r = client.chat.completions.create(model=MODEL, max_tokens=200 if is_reasoning(MODEL) else 5,
                                   messages=[{"role": "user", "content": "Reply with OK."}])
print(f"{MODEL}: {r.choices[0].message.content!r}")

deepseek-ai/DeepSeek-V4.1-Flash: 'OK'


## 5. Gold examples (few-shot)
Sent to the model as earlier chat turns, so they guide the style without appearing in the output. All examples are on non-test topics (UN resolution, US–China trade, EU sanctions).
- `GOLD_SOURCES_UR`: short Urdu news texts (turned into fact lists in section 6, like real inputs).
- `GOLD_DEPTH_UR`: the same texts rewritten at each depth, **in the same order**.
- `GOLD_EASY_META_UR`: for each easy example, the terms it explains and whether it has an everyday example.
- `GOLD_DEPTH_EN`: English versions of the depth examples (used to teach Urdu → English).
- `GOLD_UR_URDISH`: (English, Urdu, Urdish, depth) examples for the Urdish step.

In [6]:
GOLD_SOURCES_UR = [
    "اقوام متحدہ کی سلامتی کونسل نے ایک قرارداد منظور کی ہے جس میں فریقین سے فوری جنگ بندی کا مطالبہ کیا گیا ہے۔ تیرہ ارکان نے حق میں ووٹ دیا جبکہ دو نے ووٹ نہیں دیا۔ سفارتی ماہرین کا کہنا ہے کہ اس کا اثر اس بات پر منحصر ہے کہ بڑی طاقتیں اس کی کتنی حمایت کرتی ہیں۔",
    "امریکہ اور چین کے درمیان تجارتی مذاکرات کا نیا دور اگلے ہفتے شروع ہو رہا ہے۔ دونوں ممالک محصولات اور برآمدی پابندیوں پر اختلافات کم کرنے کی کوشش کریں گے۔ مبصرین کا کہنا ہے کہ کسی بڑی پیش رفت کا امکان کم ہے، تاہم مذاکرات کا جاری رہنا ایک مثبت اشارہ ہے۔",
    "یورپی یونین نے ایک ملک پر نئی اقتصادی پابندیوں کا اعلان کیا ہے۔ اس کے کچھ بینکوں کے اثاثے منجمد کیے جائیں گے اور توانائی کی برآمدات محدود کی جائیں گی۔ متاثرہ ملک نے اس فیصلے کو ناانصافی قرار دیا اور جوابی اقدامات کی دھمکی دی۔",
]

GOLD_DEPTH_UR = {
    "easy": [
        "اقوام متحدہ کی سلامتی کونسل نے ایک قرارداد منظور کی ہے۔ قرارداد ایک باقاعدہ فیصلہ ہوتا ہے۔ سلامتی کونسل اقوام متحدہ کا وہ حصہ ہے جس کا کام دنیا میں امن قائم رکھنا ہے۔ اس قرارداد میں لڑائی کے فریقوں سے کہا گیا ہے کہ وہ فوراً لڑائی بند کریں۔ اسے جنگ بندی کہتے ہیں۔ تیرہ ارکان نے اس کے حق میں ووٹ دیا اور دو نے ووٹ نہیں دیا۔ سفارت کاری، یعنی ملکوں کے آپس کے معاملات، کے ماہرین کہتے ہیں کہ یہ فیصلہ تب ہی کام کرے گا جب بڑی طاقتیں اس کا ساتھ دیں۔",
        "امریکہ اور چین اگلے ہفتے تجارتی بات چیت کا نیا دور شروع کریں گے۔ تجارتی بات چیت میں ملک آپس میں چیزیں خریدنے اور بیچنے کے اصولوں پر بات کرتے ہیں۔ دونوں ملکوں کا محصولات پر اختلاف ہے۔ محصولات وہ اضافی ٹیکس ہیں جو دوسرے ملکوں سے آنے والی چیزوں پر لگتے ہیں۔ ان کا برآمدی پابندیوں پر بھی اختلاف ہے، یعنی اس بات کی حد کہ کوئی ملک باہر کیا بیچ سکتا ہے۔ مثال کے طور پر، اگر محصول کی وجہ سے باہر سے آنے والی چاکلیٹ مہنگی ہو جائے تو لوگ مقامی چاکلیٹ خریدنے لگتے ہیں۔ ماہرین کسی بڑے معاہدے کی توقع نہیں کرتے۔ لیکن ان کا کہنا ہے کہ دونوں ملکوں کا بات چیت جاری رکھنا اچھی بات ہے۔",
        "یورپی یونین نے ایک ملک پر نئی اقتصادی پابندیوں کا اعلان کیا ہے۔ یورپی یونین یورپ کے ملکوں کا ایک گروپ ہے جو بہت سے فیصلے مل کر کرتے ہیں۔ پابندیاں تجارت اور پیسے پر لگائی جانے والی سزائیں ہیں جن سے کسی حکومت پر دباؤ ڈالا جاتا ہے۔ اس ملک کے کچھ بینکوں کے اثاثے منجمد کیے جائیں گے۔ اس کا مطلب ہے کہ یہ بینک وہ پیسہ استعمال نہیں کر سکیں گے۔ اس ملک کی توانائی کی برآمدات، یعنی وہ توانائی جو وہ باہر بیچتا ہے، بھی محدود کی جائیں گی۔ اس ملک نے فیصلے کو ناانصافی کہا۔ اس نے جوابی اقدامات کی دھمکی بھی دی، یعنی وہ بدلے میں اپنے قدم اٹھا سکتا ہے۔",
    ],
    "medium": [
        "اقوام متحدہ کی سلامتی کونسل، جو دنیا میں امن اور سلامتی کی ذمہ دار ہے، نے ایک قرارداد منظور کی ہے جس میں فریقین سے فوری جنگ بندی کا کہا گیا ہے۔ تیرہ ارکان نے حق میں ووٹ دیا اور دو نے ووٹ نہیں دیا۔ سفارتی ماہرین کہتے ہیں کہ اس کا اثر اس پر منحصر ہوگا کہ بڑی طاقتیں اس کی کتنی حمایت کرتی ہیں۔",
        "امریکہ اور چین کے درمیان تجارتی مذاکرات کا نیا دور اگلے ہفتے شروع ہوگا۔ دونوں ملک محصولات، یعنی باہر سے آنے والی چیزوں پر ٹیکس، اور برآمدی پابندیوں پر اپنے اختلافات کم کرنے کی کوشش کریں گے۔ مبصرین کو کسی بڑی پیش رفت کی امید نہیں، لیکن وہ مذاکرات کے جاری رہنے کو اچھی علامت سمجھتے ہیں۔",
        "یورپی یونین نے ایک ملک پر نئی اقتصادی پابندیوں کا اعلان کیا ہے، یعنی تجارت اور پیسے کے لین دین پر سزائیں۔ اس کے کچھ بینکوں کے اثاثے منجمد کیے جائیں گے اور اس کی توانائی کی برآمدات محدود ہوں گی۔ متاثرہ ملک نے اس فیصلے کو ناانصافی کہا اور جوابی اقدامات کی دھمکی دی۔",
    ],
    "hard": [
        "اقوام متحدہ کی سلامتی کونسل نے فریقین سے فوری جنگ بندی کا مطالبہ کرنے والی قرارداد منظور کر لی ہے۔ تیرہ ارکان نے حق میں ووٹ دیا اور دو نے ووٹ نہیں دیا۔ سفارتی ماہرین کے مطابق اس کا اثر اس پر منحصر ہوگا کہ بڑی طاقتیں اس کی کتنی حمایت کرتی ہیں۔",
        "امریکہ اور چین کے تجارتی مذاکرات کا نیا دور اگلے ہفتے شروع ہو رہا ہے، جس میں محصولات اور برآمدی پابندیوں پر اختلافات کم کرنے کی کوشش ہوگی۔ مبصرین کے مطابق بڑی پیش رفت کا امکان کم ہے، لیکن مذاکرات کا جاری رہنا خود ایک مثبت اشارہ ہے۔",
        "یورپی یونین نے ایک ملک پر اقتصادی پابندیوں کے نئے پیکج کا اعلان کیا ہے۔ اس کے تحت کچھ بینکوں کے اثاثے منجمد اور توانائی کی برآمدات محدود کی جائیں گی۔ متاثرہ حکومت نے اسے ناانصافی قرار دیا اور جوابی اقدامات کی دھمکی دی۔",
    ],
}

GOLD_EASY_META_UR = [
    ("سلامتی کونسل، قرارداد، جنگ بندی", "no"),
    ("تجارتی بات چیت، محصولات، برآمدی پابندیاں", "yes"),
    ("پابندیاں، منجمد اثاثے، جوابی اقدامات", "no"),
]

GOLD_DEPTH_EN = {
    "easy": [
        "The UN Security Council has passed a resolution. A resolution is a formal decision. The Security Council is the part of the United Nations that is responsible for keeping peace in the world. The resolution asks the sides in a conflict to stop fighting immediately. This is called a ceasefire. Thirteen members voted for it, and two did not vote. Experts on diplomacy, which means how countries deal with each other, say the decision will only work if the major powers support it. This is because the Security Council cannot easily force countries to follow its decisions.",
        "The US and China will start a new round of trade talks next week. In trade talks, countries discuss the rules for buying and selling goods with each other. The two countries disagree about tariffs. Tariffs are extra taxes on goods that come from other countries. They also disagree about export restrictions, which are limits on what a country can sell abroad. For example, if a tariff makes imported chocolate more expensive, shoppers may buy local chocolate instead. Experts do not expect a big agreement. But they say it is a good sign that the two sides keep talking.",
        "The European Union has announced new economic sanctions on a country. The European Union is a group of European countries that make many decisions together. Sanctions are penalties on trade and money that are used to pressure a government. Some of the country's banks will have their assets frozen. This means the banks cannot use that money. The country's energy exports, meaning the energy it sells abroad, will also be limited. The country called the decision unfair. It also threatened countermeasures, which means it may take steps to hit back.",
    ],
    "medium": [
        "The UN Security Council, the UN body responsible for international peace and security, has passed a resolution calling on the parties for an immediate ceasefire. Thirteen members voted in favour and two did not vote. Diplomatic experts say the resolution's impact will depend on how strongly the major powers support it.",
        "A new round of trade talks between the United States and China begins next week. Both countries will try to narrow their differences over tariffs, which are taxes on imported goods, and over export restrictions. Observers say a major breakthrough is unlikely, but they see the continuation of the talks as a positive sign.",
        "The European Union has announced new economic sanctions, meaning penalties on trade and finance, against a country. The assets of some of its banks will be frozen and its energy exports will be limited. The affected country called the decision unfair and threatened countermeasures.",
    ],
    "hard": [
        "The UN Security Council has adopted a resolution calling on the parties to an immediate ceasefire, with thirteen members voting in favour and two not voting. Diplomatic experts say its impact will depend on how strongly the major powers back it.",
        "A new round of US-China trade talks begins next week, aimed at narrowing differences over tariffs and export restrictions. Observers consider a major breakthrough unlikely, but read the continuation of the dialogue itself as a positive signal.",
        "The European Union has announced a new package of economic sanctions against a country. It will freeze the assets of some of its banks and limit its energy exports. The targeted government called the decision unfair and threatened countermeasures.",
    ],
}

# (English, Urdu, Urdish, depth)
GOLD_UR_URDISH = [
    ("The UN Security Council has passed a resolution calling on the parties for an immediate ceasefire. Thirteen members voted in favour and two did not vote. Diplomatic experts say its impact depends on how strongly major powers support it.",
     "اقوام متحدہ کی سلامتی کونسل نے ایک قرارداد منظور کی ہے جس میں فریقین سے فوری جنگ بندی کا مطالبہ کیا گیا ہے۔ قرارداد کے حق میں تیرہ ووٹ آئے جبکہ دو ارکان نے ووٹنگ میں حصہ نہیں لیا۔ سفارتی ماہرین کے مطابق اس قرارداد پر عمل درآمد کا انحصار اس بات پر ہے کہ بڑی طاقتیں اس کی کتنی حمایت کرتی ہیں۔",
     "UN Security Council ne ek resolution pass ki hai jis mein parties se foran ceasefire ka mutalba kiya gaya hai. Resolution ke haq mein terah votes aaye jabke do members ne voting mein hissa nahi liya. Diplomatic experts ke mutabiq is resolution par amal hona is baat par depend karta hai ke bari powers ise kitna support karti hain.",
     "medium"),
    ("The UN Security Council is the UN group that decides on peace and security. It passed a resolution, which is a formal decision, asking both sides to stop fighting right away. This is called a ceasefire. Thirteen members voted yes and two chose not to vote. Experts say the decision will only work if the big powers support it.",
     "اقوام متحدہ کی سلامتی کونسل اقوام متحدہ کا وہ ادارہ ہے جو امن اور سلامتی کے فیصلے کرتا ہے۔ اس نے ایک قرارداد منظور کی، یعنی ایک باقاعدہ فیصلہ، جس میں دونوں فریقوں سے کہا گیا کہ فوراً لڑائی بند کریں۔ اسے جنگ بندی کہتے ہیں۔ تیرہ ارکان نے حق میں ووٹ دیا اور دو نے ووٹ نہیں دیا۔ ماہرین کا کہنا ہے کہ یہ فیصلہ تب ہی کارگر ہوگا جب بڑی طاقتیں اس کی حمایت کریں گی۔",
     "UN Security Council, UN ka woh group hai jo peace aur security ke faislay karta hai. Is ne ek resolution pass ki hai, yaani ek formal faisla, jis mein dono sides se kaha gaya hai ke foran larai band karein. Isay ceasefire kehte hain. Terah members ne haan mein vote diya aur do ne vote hi nahi diya. Experts kehte hain ke yeh faisla tabhi kaam karega jab bari powers iska saath dein.",
     "easy"),
]

for d in DEPTHS:
    assert len(GOLD_DEPTH_UR[d]) == len(GOLD_SOURCES_UR) == len(GOLD_DEPTH_EN[d]), f"gold '{d}' lists must have the same length"
assert len(GOLD_EASY_META_UR) == len(GOLD_SOURCES_UR)
print("Gold examples OK:", len(GOLD_SOURCES_UR), "depth examples,", len(GOLD_UR_URDISH), "Urdish examples")

Gold examples OK: 3 depth examples, 2 Urdish examples


## 6. Prompts
Each request is a chat: rules in the system message, each gold example as a separate user → assistant turn, the real text as the last user message.

In [7]:
# ---------- step 2: input -> Urdu fact list + terms ----------
SYS_PREP = """You summarise news articles (in Urdu or English) in Urdu.
Reply in exactly this format and nothing else:
SUMMARY: a list of 4 to 8 short facts from the article (fewer for a short text, more for a long one), one per line, each starting with "- ". Each fact is one short, simple sentence in everyday Urdu (Urdu script). Keep every name, number and date exactly as in the article; copy each person's name from the article and never guess or merge names. If the article is in English, write names in their usual Urdu spelling. Write numbers as digits. Add nothing that is not in the article.
TERMS: 0 to 3 political, legal, economic or international-relations concepts from your summary whose meaning a beginner could not guess from the words themselves (for example: گورنر راج، پابندیاں، حلقہ انتخاب). Write each exactly as it appears in your summary. Never pick ordinary words (احتجاج، پولیس، زخمی). Separate them with commas. Write none if there are none."""

# ---------- step 3: facts -> Urdu depths ----------
SYS_DEPTH_RULES = {
    "easy": """You explain news to a reader who knows nothing about politics, government or international relations.
Rules:
- One paragraph of plain, natural prose with short sentences: one main idea per sentence, most sentences under 20 words.
- Use everyday words instead of formal ones (for example: "ratified" becomes "officially approved", "bilateral" becomes "between the two countries").
- Keep every fact of the Text: every name, place, number and date, with the same meaning. Never make a fact stronger or weaker (for example, "delayed" must not become "cancelled"). Do not add facts, guesses or conclusions about the event.
- Explain difficult political, legal, economic or international concepts a beginner would not know, including the listed Terms, once, where they first appear: either as a short phrase, or in the very next sentence ("This means ..."). Never put two explanations in one sentence, and never collect explanations at the end. Skip any Term whose everyday words already make its meaning clear.
- The first time an IR actor appears, say briefly who they are, but only if you are sure (for example: "NATO, a military alliance of North American and European countries"). If you are not sure, just use the name. Never explain things almost everyone knows (for example: never say that a country is a country).
- You may add ONE short sentence of general background that helps a beginner understand the main concept (for example: "In many countries, parliament must approve a treaty before it takes effect."). It must be general knowledge, never a new fact about this event.
- If Example is yes, add one short everyday example of one of the Terms. If Example is no, add no example.
- Never spell out an abbreviation unless the Text spells it out.
- Rewrite ONLY the Text in the latest message. Output only the rewrite, with no notes or comments.""",
    "medium": """You explain news to a reader with basic or intermediate knowledge of international relations: they know common terms but not specialist ones. Write in simple, easy-to-read language.
Rules:
- One paragraph of clear, natural prose.
- Use the correct political and legal terms. Explain the 1 or 2 most important or difficult IR or political concepts in the Text, each in a short phrase where it first appears. Do not explain common ones (government, election, protest, minister).
- Identify an actor briefly only where it helps understanding (for example: "the WTO, the body that oversees global trade rules").
- Keep every fact of the Text: every name, place, number and date, with the same meaning. Do not add facts, descriptions, or names of organisations, groups or countries that are not in the Text.
- Never spell out an abbreviation unless the Text spells it out.
- Rewrite ONLY the Text in the latest message. Output only the rewrite, with no notes or comments.""",
    "hard": """You write news for a reader who is familiar with international relations concepts and terms.
Rules:
- One paragraph of clear, simple prose that uses the correct technical IR and legal terms without explaining them. The language must stay easy to read: only the terms are technical, not the sentences.
- Keep official and legal terms exactly as in the Text; do not swap them for near-synonyms (for example: "postponed" is not "cancelled", "sanctions" is not "embargo", "did not vote" is not "abstained").
- Never add facts, motives, predictions, consequences or interpretations that the Text does not state. Avoid speculative words (potentially, may indicate, likely, signalling, in a move to).
- Precise does not mean inflated: prefer plain verbs (said, led, began, visited) over showy words (spearheaded, delineated, populace, modalities).
- Keep every fact of the Text: every name, place, number and date. About the same length as the Text; never compress it into one or two sentences.
- Never spell out an abbreviation unless the Text spells it out.
- Rewrite ONLY the Text in the latest message. Output only the rewrite, with no notes or comments.""",
}

UR_LINE = ("- Write in simple, natural Urdu (Urdu script), the way an educated Pakistani explains news in conversation, not in heavy newspaper style. "
           "Prefer common spoken words (for example: شروع instead of آغاز، مدد instead of معاونت، اس کے علاوہ instead of علاوہ ازیں). "
           "English words that are common in everyday Urdu are fine. The depth changes how much IR knowledge is assumed, never how difficult the language is. Write numbers as digits.\n"
           "- The Text is a list of facts. Turn it into one connected paragraph for this reader, in your own words. Do not copy the facts line by line.\n")
SYS_DEPTH = {d: SYS_DEPTH_RULES[d].replace("- Rewrite ONLY the Text", UR_LINE + "- Rewrite ONLY the Text") for d in DEPTHS}
SYS_DEPTH["easy"] = SYS_DEPTH["easy"].replace(UR_LINE, UR_LINE + "- Use simple, everyday spoken Urdu words, not bookish ones. If the Text uses a formal newspaper word, say it in everyday words instead.\n")

def as_facts(text):
    """Turn a prose gold source into a fact list, so gold inputs look like real inputs."""
    return "\n".join("- " + s.strip() + "۔" for s in text.split("۔") if s.strip())

def depth_user(depth, text, terms="none", ex="no"):
    if depth == "easy":
        return f"Text: {text}\nTerms: {terms}\nExample: {ex}"
    return f"Text: {text}"

def depth_messages(depth, facts, terms, ex):
    msgs = [{"role": "system", "content": SYS_DEPTH[depth]}]
    for i, s in enumerate(GOLD_SOURCES_UR):
        t, e = GOLD_EASY_META_UR[i] if depth == "easy" else ("none", "no")
        msgs.append({"role": "user", "content": depth_user(depth, as_facts(s), t, e)})
        msgs.append({"role": "assistant", "content": GOLD_DEPTH_UR[depth][i]})
    msgs.append({"role": "user", "content": depth_user(depth, facts, terms, ex)})
    return msgs

# ---------- step 4: Urdu -> English ----------
READER = {
    "easy":   "a reader with no IR knowledge: use simple, everyday words and short sentences; describe actions in plain words instead of formal nouns",
    "medium": "a reader with some IR knowledge: use simple, clear words",
    "hard":   "a reader familiar with IR terms: keep the technical terms, but use clear, simple sentences",
}

SYS_UR_TO_EN = """You translate Urdu text (Urdu script) into clear, natural English.
Rules:
- Same content, same order, same explanations. Do not add or drop anything.
- Write for the Reader given in the message. Never make the English more formal than needed: prefer plain words (for example: "began" rather than "commenced", "use" rather than "utilize").
- Keep every name, number and date. Use standard English spellings of names and places.
- Keep the exact meaning of official actions (for example: "appointed" is not "elected", "suspended" is not "dismissed"), and never add names or details that are not in the Urdu text.
- Output only the English translation."""

GOLD_UR_EN = [(GOLD_DEPTH_UR["easy"][0], GOLD_DEPTH_EN["easy"][0], "easy"),
              (GOLD_DEPTH_UR["medium"][1], GOLD_DEPTH_EN["medium"][1], "medium"),
              (GOLD_DEPTH_UR["hard"][2], GOLD_DEPTH_EN["hard"][2], "hard")]

def ur_to_en_messages(urdu, depth, original_en=""):
    msgs = [{"role": "system", "content": SYS_UR_TO_EN}]
    for ur, en, d in GOLD_UR_EN:
        msgs.append({"role": "user", "content": f"Reader: {READER[d]}\n\n{ur}"})
        msgs.append({"role": "assistant", "content": en})
    ref = (f"Original English article (only for spellings of names, places and terms):\n{original_en}\n\n"
           if original_en else "")
    msgs.append({"role": "user", "content": f"{ref}Reader: {READER[depth]}\n\n{urdu}"})
    return msgs

# ---------- step 5: Urdu + English -> Urdish ----------
SYS_URDISH = """You write Urdish: Urdu written in English (Roman) letters, mixed with common English words, the way an educated Pakistani writes in WhatsApp messages.
You get an Urdu text and its English translation. They say the same thing.
Rules:
- Follow the Urdu text sentence by sentence. Do not add, drop or change any fact, name or number.
- Write for the Reader given in the message, the way people actually text: where the Urdu uses a formal or bookish word, use the everyday word or the common English word instead (for example: log instead of afrad, shuru instead of aaghaz, madad instead of muawanat).
- Keep news and technical terms in English the way people text them (protest, government, sanctions, ceasefire, governor's rule).
- Use Pakistani Urdu words, never Hindi words (sarkar, shanti, chinta, vishay, adhikari, samasya, jaanch).
- Spell names exactly as in the English translation. No Urdu script and no Devanagari. Keep numbers as digits.
- Output only the Urdish text."""

def urdish_user(urdu, english, depth):
    return f"Reader: {READER[depth]}\n\nUrdu text:\n{urdu}\n\nEnglish translation:\n{english}"

def urdish_messages(urdu, english, depth):
    msgs = [{"role": "system", "content": SYS_URDISH}]
    for en, ur, rd, d in GOLD_UR_URDISH:
        msgs.append({"role": "user", "content": urdish_user(ur, en, d)})
        msgs.append({"role": "assistant", "content": rd})
    msgs.append({"role": "user", "content": urdish_user(urdu, english, depth)})
    return msgs

# Sentences from the gold examples. If one shows up in an output, the model copied an example.
GOLD_SENTS = {s.strip().lower()
              for txt in [*GOLD_SOURCES_UR, *sum(GOLD_DEPTH_UR.values(), []), *sum(GOLD_DEPTH_EN.values(), []),
                          *[t for ex in GOLD_UR_URDISH for t in ex[:3]]]
              for s in re.split(r"[.!?۔]", txt) if len(s.split()) >= 6}
print("Prompts ready")

Prompts ready


## 7. Load the articles
Reads every `.jsonl`, `.json` and `.txt` file in each folder of `ARTICLES_DIRS`, and tags each article as Urdu or English.

In [8]:
def make_article_id(url, text):
    key = url.strip() if url else text.strip()[:2000]
    return hashlib.sha256(key.encode("utf-8")).hexdigest()

def _as_article(d, fallback_url):
    text = d.get("text") or d.get("content") or d.get("body") or ""
    url = d.get("url") or d.get("link") or fallback_url
    return {"url": url, "text": text, "title": d.get("title", "")}

def load_articles(folder):
    arts = []
    for path in sorted(glob.glob(os.path.join(folder, "**", "*"), recursive=True)):
        ext = os.path.splitext(path)[1].lower()
        rel = os.path.relpath(path, folder)
        try:
            if ext == ".jsonl":
                for n, line in enumerate(open(path, encoding="utf-8")):
                    if line.strip():
                        arts.append(_as_article(json.loads(line), f"{rel}#{n}"))
            elif ext == ".json":
                data = json.load(open(path, encoding="utf-8"))
                for n, d in enumerate(data if isinstance(data, list) else [data]):
                    arts.append(_as_article(d, f"{rel}#{n}"))
            elif ext == ".txt":
                arts.append({"url": rel, "text": open(path, encoding="utf-8").read(), "title": ""})
        except Exception as e:
            print(f"  skipped {rel}: {type(e).__name__}: {e}")
    out, seen = [], set()
    for a in arts:
        if not a["text"].strip():
            continue
        a["article_id"] = make_article_id(a["url"], a["text"])
        if a["article_id"] in seen:
            continue
        seen.add(a["article_id"])
        if NUM_SHARDS > 1 and int(a["article_id"][:8], 16) % NUM_SHARDS != SHARD_INDEX:
            continue
        out.append(a)
    return out[:MAX_ARTICLES] if MAX_ARTICLES else out

ARTICLES = []
for d in ARTICLES_DIRS:
    assert os.path.isdir(d), f"Folder not found: {d}"
    ARTICLES += load_articles(d)
for a in ARTICLES:   # Urdu if at least 30% of the characters are Urdu script
    a["src_lang"] = "ur" if len(re.findall(r"[\u0600-\u06FF]", a["text"])) > 0.3 * len(a["text"]) else "en"
print(len(ARTICLES), "articles loaded |", dict(collections.Counter(a["src_lang"] for a in ARTICLES)))

80 articles loaded | {'ur': 40, 'en': 40}


## 8. Helpers
LLM calls, automatic checks (one retry if something is clearly wrong), and the input builder: cleaned paragraphs, 150–900 words, sometimes starting mid-article.

In [9]:
from ftfy import fix_text

LOCK = threading.Lock()

def clean(t):
    t = re.sub(r"<think>.*?</think>", "", t or "", flags=re.S).strip()
    t = re.sub(r"^\s*(Urdish|Urdu|English|Rewrite|Output|Text)\s*:\s*", "", t, flags=re.I)
    t = t.replace("*", "")
    return re.sub(r"\s*\n+\s*", " ", t.strip().strip('"')).strip()

def llm(messages, max_tokens=900):
    mt = max_tokens * 8 if is_reasoning(MODEL) else max_tokens
    for attempt in range(5):
        try:
            r = client.chat.completions.create(model=MODEL, max_tokens=mt, temperature=TEMPERATURE, messages=messages)
            c = r.choices[0].message.content or ""
            if not c.strip():
                print(f"   [empty] finish={r.choices[0].finish_reason} | usage={getattr(r, 'usage', None)}")
            return c
        except Exception as e:
            print(f"   [llm retry {attempt+1}] {type(e).__name__}: {str(e)[:80]}")
            time.sleep(2 ** attempt)
    raise RuntimeError("llm failed after retries")

def copied_gold(t):
    tl = t.lower()
    return [s for s in GOLD_SENTS if s in tl]

def remove_gold(t):
    """Last resort: drop any copied example sentence from the output."""
    parts = re.split(r"(?<=[.!?۔])\s+", t)
    return " ".join(p for p in parts if not copied_gold(p)).strip()

DEVANAGARI = re.compile(r"[\u0900-\u097F]")
HINDI_WORDS = re.compile(r"\b(sarkar|shanti|shant|vishay|adhikari|hinsak|hinsa|samasya|upyog|rashtra|suraksha|yuddh|mantri|pradhan|chinta|jaanch|kripya|dhanyavad)\b", re.I)
URDU_FUNCTION_WORDS = {"hai", "hain", "ka", "ki", "ke", "mein", "ne", "ko", "se", "aur", "tha", "thi", "thay",
                       "hua", "hue", "kiya", "gaya", "gayi", "nahi", "bhi", "yeh", "woh"}

def problem(t, kind, source=""):
    """What is wrong with an output, or None. Only used to ask the model ONE more time."""
    if not t:
        return "Your answer was empty."
    if copied_gold(t):
        return "You copied the earlier examples. Give the answer for my latest text only."
    if re.search(r"\bnote\s*:|\b(is|are) not (present|mentioned|found|given)\b|\bthe (text|summary) (says|does|mentions|states)", t, re.I):
        return "Your answer contains a comment about the text. Output only the rewrite itself."
    if kind in DEPTHS and source:
        dig = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")
        extra = set(re.findall(r"[0-9]+", t.translate(dig))) - set(re.findall(r"[0-9]+", source.translate(dig)))
        extra = {x for x in extra if len(x) > 1}   # single digits are often written as words in the source
        if extra:
            return f"Your answer contains numbers that are not in the Text ({', '.join(sorted(extra))}). Remove anything that is not in the Text."
    if DEVANAGARI.search(t):
        return "You used Hindi (Devanagari) letters. Write it again without any Devanagari."
    if kind == "urdish":
        words = re.findall(r"[a-z']+", t.lower())
        if words and sum(w in URDU_FUNCTION_WORDS for w in words) / len(words) < 0.08:
            return "Your answer is in English. Write it in Urdish: Urdu sentences in Roman letters, with only some English words mixed in."
        if source and len(t.split()) < 0.7 * len(source.split()):
            return "You left out some sentences. Convert every sentence of the Urdu text."
        if HINDI_WORDS.search(t):
            return f"You used the Hindi word '{HINDI_WORDS.search(t).group(0)}'. Use Pakistani Urdu words instead."
    if kind in ("medium", "hard") and source and len(t.split()) < 0.6 * len(source.split()):
        return "Your answer left out facts. Cover every sentence of the Text, with every name, place and number."
    if kind == "easy" and source and len(t.split()) < 0.8 * len(source.split()):
        return "Your answer left out facts. Cover every sentence of the Text, with every name, place and number."
    return None

def generate(messages, kind, source="", max_tokens=900):
    """One call; if something is clearly wrong, ask once more with the reason. The output is always kept."""
    t = clean(llm(messages, max_tokens))
    why = problem(t, kind, source)
    if why:
        print(f"   [retry {kind}] {why[:90]}")
        t2 = clean(llm(messages + [{"role": "assistant", "content": t or "(empty)"},
                                    {"role": "user", "content": why}], max_tokens))
        still = problem(t2, kind, source) if t2 else "empty"
        if still:
            print(f"   [still wrong {kind}] {still[:90]}")
        if t2 and (still is None or not t):
            t = t2
    return remove_gold(t) if copied_gold(t) else t

def parse_prep(t):
    """Fact list (kept as lines) and terms from the summary reply."""
    t = re.sub(r"<think>.*?</think>", "", t or "", flags=re.S).replace("**", "")
    m = re.search(r"SUMMARY\s*:\s*(.*?)(?:\n\s*TERMS\s*:|$)", t, re.S | re.I)
    raw = (m.group(1) if m else t).replace("*", "")
    facts = "\n".join(l.strip() for l in raw.strip().splitlines() if l.strip())
    m = re.search(r"TERMS\s*:\s*(.*)", t, re.I)
    terms = [x.strip(" .۔'\"") for x in re.split(r"[،,]", m.group(1) if m else "")]
    terms = [x for x in terms if x and x.lower() not in ("none", "n/a", "کوئی نہیں") and x in facts][:3]
    return facts, terms

# ---------- training input: cleaned paragraphs of varying length ----------
TAIL_MARKERS = ["Watch our pick of standout clips", "This breaking news story is being updated",
                "Get our flagship newsletter", "Copyright ©", "You can receive Breaking News"]

def clean_paragraphs(text):
    """Article paragraphs only: no page tail, captions, link lines or promo lines."""
    text = fix_text(text)
    for marker in TAIL_MARKERS:
        i = text.find(marker)
        if i > 0:
            text = text[:i]
    paras = []
    for p in text.split("\n"):
        p = p.strip()
        if len(p.split()) < 6:                                     # headings, bylines, short link lines
            continue
        if not re.search(r"""[.!?۔؟:"”’‘')»]$""", p):               # captions and headlines have no final punctuation
            continue
        if re.search(r"BBC|بی بی سی", p) and len(p.split()) < 15:  # short promo lines (keeps "...told the BBC")
            continue
        paras.append(p)
    return paras

def article_input(art):
    """Whole paragraphs, INPUT_MIN-INPUT_MAX words, ~30% starting mid-article. Same result on every run."""
    paras = clean_paragraphs(art["text"])
    target = INPUT_MIN + int(art["article_id"][8:16], 16) % (INPUT_MAX - INPUT_MIN + 1)
    h = int(art["article_id"][16:24], 16)
    start = 0
    if h % 10 < 3 and len(paras) > 4:                             # like a chunk retrieved from the middle
        start = 1 + (h // 10) % (len(paras) // 2)
    out, n = [], 0
    for p in paras[start:]:
        if n >= target:
            break
        out.append(p)
        n += len(p.split())
    return "\n\n".join(out) if n >= 30 else None

print("Helpers ready")

Helpers ready


## 9. Pipeline: one article → 9 cells
1. Build the input (cleaned paragraphs).
2. Input → Urdu facts + terms (cached).
3. For each depth: Urdu cell → English cell → Urdish cell.

An article is only skipped if it has no usable text, or if the API keeps failing (those are retried on the next run of section 11).

In [10]:
PREP_CACHE = {}
if os.path.exists(PREP_PATH):
    for line in open(PREP_PATH, encoding="utf-8"):
        r = json.loads(line)
        PREP_CACHE[r.pop("article_id")] = r

def prep(art, source):
    aid = art["article_id"]
    if aid in PREP_CACHE:
        return PREP_CACHE[aid]
    facts, terms = parse_prep(llm([{"role": "system", "content": SYS_PREP},
                                   {"role": "user", "content": source}], 1600))
    if not facts.strip():
        return {"reject": "empty_summary"}   # not cached, so it is tried again next time
    res = {"facts": facts, "terms": terms}
    with LOCK:
        PREP_CACHE[aid] = res
        with open(PREP_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps({"article_id": aid, **res}, ensure_ascii=False) + "\n")
    return res

def make(art):
    base = {"article_id": art["article_id"], "url": art["url"], "machine": MACHINE_ID,
            "src_lang": art.get("src_lang", "ur")}
    source = article_input(art)
    if not source:
        return {**base, "reject": "no_text"}
    p = prep(art, source)
    if p.get("reject"):
        return {**base, "reject": p["reject"]}
    use_ex = int(art["article_id"][:8], 16) % 100 < EXAMPLE_RATE * 100
    row = {**base, "source": source, "facts": p["facts"], "terms": p["terms"],
           "use_example": use_ex, "cells": {}, "reject": None}
    orig_en = source if base["src_lang"] == "en" else ""
    for depth in DEPTHS:
        ur = generate(depth_messages(depth, p["facts"], "، ".join(p["terms"]) or "none", "yes" if use_ex else "no"),
                      kind=depth, source=p["facts"], max_tokens=1400)
        if not ur:
            return {**base, "reject": f"empty_ur_{depth}"}
        en = generate(ur_to_en_messages(ur, depth, orig_en), kind="en", max_tokens=1100)
        if not en:
            return {**base, "reject": f"empty_en_{depth}"}
        row["cells"][f"ur_{depth}"] = ur
        row["cells"][f"en_{depth}"] = en
        row["cells"][f"urdish_{depth}"] = generate(urdish_messages(ur, en, depth), kind="urdish",
                                                   source=en, max_tokens=1100)
    return row

print("Pipeline ready | model:", MODEL)

Pipeline ready | model: deepseek-ai/DeepSeek-V4.1-Flash


## 10. Pilot: a few articles, nothing saved
By default: the first 2 English articles and the first Urdu article. Check that the input is clean (no headline lists, captions or broken characters), the facts are accurate, names match the original, and the depths differ.

In [11]:
PILOT_IDS = ([i for i, a in enumerate(ARTICLES) if a["src_lang"] == "en"][:2] +
             [i for i, a in enumerate(ARTICLES) if a["src_lang"] == "ur"][:1])

for i in PILOT_IDS:
    row = make(ARTICLES[i])
    print("=" * 100)
    print(f"[{i}] {row['url']} | source: {row['src_lang']} | skipped: {row.get('reject')}")
    if row.get("facts"):
        print(f"\nINPUT ({len(row['source'].split())} words):\n{row['source'][:600]} ...")
        print("\nFACTS:\n" + row["facts"], "\nTERMS:", row["terms"], "| example:", row["use_example"])
        for d in DEPTHS:
            for l in LANGS:
                print(f"\n[{l} / {d}]\n{row['cells'][f'{l}_{d}']}")

   [retry urdish] You used the Hindi word 'jaanch'. Use Pakistani Urdu words instead.
[40] https://www.bbc.co.uk/news/articles/crl7yjlpx2po?at_medium=RSS&at_campaign=rss | source: en | skipped: None

INPUT (207 words):
The Israeli army has ordered a criminal investigation into the killing in Gaza of six-year-old Hind Rajab, whose death was internationally condemned after a desperate recording of her speaking to paramedics was broadcast around the world.

The Palestinian child was one of seven family members travelling in a vehicle that came under Israeli fire in Gaza City in January 2024.

She initially survived the attack that killed several family members, but her body was recovered days later along with her relatives and two paramedics who had tried to save her in an ambulance.

In a statement, the Israel ...

FACTS:
- اسرائیلی فوج نے غزہ میں 6 سالہ ہند رجب کے قتل کی مجرمانہ تحقیقات کا حکم دیا ہے۔
- ہند رجب جنوری 2024 میں غزہ شہر میں اسرائیلی فائرنگ کا نشانہ بننے والی گاڑی میں سوار 

## 11. Full run (resumable)
Every processed article is appended to `grid_raw_v3_<machine>.jsonl`. If the session dies, run this cell again: it continues, and also retries articles that failed with an API error or an empty output.

In [ ]:
done = {}
if os.path.exists(RAW_PATH):
    for line in open(RAW_PATH, encoding="utf-8"):
        r = json.loads(line)
        if not str(r.get("reject") or "").startswith(("error", "empty")):   # API errors and empty outputs are retried
            done[r["article_id"]] = r

todo = [a for a in ARTICLES if a["article_id"] not in done]
print(f"{len(ARTICLES)} articles | {len(done)} already done | {len(todo)} to process | model = {MODEL}")

def safe(art):
    try:
        return make(art)
    except Exception as e:
        return {"article_id": art["article_id"], "url": art["url"], "machine": MACHINE_ID,
                "src_lang": art.get("src_lang", "ur"), "reject": f"error:{type(e).__name__}"}

B = WORKERS * 4
t0 = time.time()
with ThreadPoolExecutor(WORKERS) as ex, open(RAW_PATH, "a", encoding="utf-8") as f:
    for i in range(0, len(todo), B):
        for r in ex.map(safe, todo[i:i + B]):
            done[r["article_id"]] = r
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
        f.flush()
        counts = collections.Counter(r.get("reject") or "done" for r in done.values())
        print(f"  {min(i + B, len(todo))}/{len(todo)} | {time.time() - t0:.0f}s | " +
              ", ".join(f"{k}={v}" for k, v in counts.most_common()))

## 12. Export this machine's dataset
Writes in `OUT_DIR`:
- `grid_<machine>.csv`: the training data, 9 rows per article. `input` = the article text the model reads; `output` = what it learns to write.
- `review_<machine>.csv`: about 10% of articles (all 9 cells) for native-speaker review, with `ok` / `notes` columns.

In [ ]:
def write_csv(path, header, rows):
    with open(path, "w", newline="", encoding="utf-8-sig") as f:   # utf-8-sig so Urdu opens correctly in Excel
        w = csv.writer(f, quoting=csv.QUOTE_ALL)
        w.writerow(header)
        w.writerows(rows)

rows = {}
for line in open(RAW_PATH, encoding="utf-8"):
    r = json.loads(line)
    rows[r["article_id"]] = r
good = [r for r in rows.values()
        if not r.get("reject") and r.get("source") and all(r["cells"].get(f"{l}_{d}") for l in LANGS for d in DEPTHS)]
skipped = collections.Counter(r.get("reject") for r in rows.values() if r.get("reject"))

grid_path = os.path.join(OUT_DIR, f"grid_{MACHINE_ID}.csv")
write_csv(grid_path,
          ["article_id", "url", "machine", "src_lang", "language", "depth", "input_words",
           "input", "facts", "output", "terms", "use_example"],
          [[r["article_id"], r["url"], r["machine"], r.get("src_lang", "ur"), l, d, len(r["source"].split()),
            r["source"], r["facts"], r["cells"][f"{l}_{d}"], "; ".join(r["terms"]), r["use_example"]]
           for r in good for d in DEPTHS for l in LANGS])

sample = random.Random(7).sample(good, min(len(good), max(5, round(len(good) * 0.1))))
write_csv(os.path.join(OUT_DIR, f"review_{MACHINE_ID}.csv"),
          ["article_id", "src_lang", "language", "depth", "input", "facts", "output", "ok (y/n)", "notes"],
          [[r["article_id"], r.get("src_lang", "ur"), l, d, r["source"], r["facts"], r["cells"][f"{l}_{d}"], "", ""]
           for r in sample for d in DEPTHS for l in LANGS])

print(f"Articles in dataset: {len(good)} -> {len(good) * 9} rows in {grid_path}")
if good:
    print("Source languages:", dict(collections.Counter(r.get("src_lang", "ur") for r in good)))
    print("Input words: min", min(len(r["source"].split()) for r in good),
          "| max", max(len(r["source"].split()) for r in good))
print("Skipped:", dict(skipped) or "none")